# Llama from Scratch

This notebook is a teaching-first walkthrough of a tiny Llama-style language model built in PyTorch.
We build the model piece by piece, explaining every innovation that separates Llama from the GPT-2 architecture.

If you have not already worked through the GPT-2 notebook in this repository, **do that first**.
This notebook assumes you understand token embeddings, causal self-attention, residual connections, and the basic transformer training loop.
We will not re-derive those ideas here; instead we focus on the three key upgrades that Llama introduces.

## Audience
- Learners who have already built a basic GPT-style model.
- Students who want to understand the modern dense transformer baseline.
- Anyone curious about RoPE, Grouped Query Attention, or SwiGLU.

## Prerequisites
- Comfortable reading the GPT-2 notebook (attention, MLP, residual connections, training loop).
- Basic PyTorch concepts such as tensors, modules, and autograd.
- Familiarity with complex numbers or 2D rotations is helpful but not required.

## Learning Goals
By the end of this notebook, you should be able to:
1. Explain how RoPE encodes position through rotation instead of addition.
2. Describe how Grouped Query Attention reduces KV-cache memory by sharing K/V heads.
3. Implement SwiGLU and explain why gating outperforms simple activations.
4. Identify the architectural differences between GPT-2 and Llama at the code level.
5. Train a tiny Llama model on Tiny Shakespeare and generate text from it.

## Outline

1. Set up the environment and imports.
2. Define the `LlamaConfig` (note the new fields: `n_kv_head`, `rope_theta`).
3. Build the model one component at a time:
   - RMSNorm (brief recap),
   - RoPE: `precompute_rope_freqs` + `apply_rope` (the big teaching moment),
   - Grouped Query Attention,
   - SwiGLU MLP,
   - LlamaBlock,
   - Full Llama model.
4. Smoke test the model (including a GQA verification).
5. Build the data pipeline and verify the tokenizer/batches.
6. Add training utilities and a full training loop.
7. Run a lightweight pipeline preview.
8. Add text generation.
9. Optional checkpoint demo.
10. Exercise: compare MHA vs GQA parameter counts.
11. Common pitfalls and next steps.

## 1. Setup

This install cell keeps the notebook easy to run in a fresh environment such as Colab.
If your environment already has these packages, rerunning this cell is harmless.

In [ ]:
!pip install torch numpy requests -q

## 2. Imports

We gather the core tools once so every later cell can stay focused on one idea.

Teaching note:
- `dataclass` gives us a clean config object.
- `Path` helps keep file paths portable between local Jupyter and Colab-style environments.
- `torch.nn` holds reusable neural network layers.
- `torch.nn.functional` contains tensor-level math functions like `softmax` and `cross_entropy`.

In [ ]:
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path

import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

### `LlamaConfig`

This dataclass collects the high-level knobs that control the model.
If you have seen the GPT-2 config, most fields will look familiar.
Two new fields capture the Llama innovations:

- `vocab_size=256` because we tokenize raw UTF-8 bytes (same as the GPT-2 notebook).
- `n_embd` is the width of the hidden representation.
- `n_head` is the number of **query** heads in attention.
- **`n_kv_head`** is the number of **key/value** heads. When `n_kv_head < n_head`, we get Grouped Query Attention. Each group of `n_head // n_kv_head` query heads shares one K and one V head. Setting `n_kv_head == n_head` recovers standard multi-head attention; setting `n_kv_head == 1` gives multi-query attention (maximum sharing).
- `n_layer` controls depth.
- `seq_len` is the maximum context window.
- `dropout=0.0` keeps the tutorial deterministic and simple.
- **`rope_theta`** is the base frequency for Rotary Position Embeddings. Higher values stretch the wavelengths, allowing the model to generalize to longer sequences. The default `10000.0` matches the original RoPE paper. Llama 3 uses `500000.0` to support 128K context windows, but for our tiny model the standard value is plenty.

In [ ]:
@dataclass
class LlamaConfig:
    vocab_size: int = 256       # byte-level tokenization (same as base model)
    n_embd: int = 64            # embedding dimension
    n_head: int = 4             # number of query heads
    n_kv_head: int = 2          # number of key/value heads (GQA: n_kv_head < n_head)
    n_layer: int = 4            # number of transformer blocks
    seq_len: int = 256          # maximum sequence length
    dropout: float = 0.0        # dropout rate
    rope_theta: float = 10000.0 # RoPE base frequency

config = LlamaConfig()
print(config)
print(f"\nGQA group size: {config.n_head // config.n_kv_head} query heads per KV head")

## 4. Model Building Blocks

We now build the Llama transformer from the inside out.
The order mirrors the final architecture:
1. RMSNorm (brief recap from the GPT-2 notebook),
2. RoPE (precomputing frequencies, applying rotations),
3. Grouped Query Attention,
4. SwiGLU MLP,
5. LlamaBlock,
6. Full Llama model.

Each section introduces exactly one of the three Llama innovations so you can study them in isolation.

### `RMSNorm`

This is identical to the GPT-2 notebook, so we keep the recap brief.

`RMSNorm` rescales a vector by its root-mean-square magnitude:

`RMSNorm(x) = x / sqrt(mean(x^2) + eps) * gamma`

It is simpler than `LayerNorm` (no mean subtraction) and appears in all modern Llama variants.
We normalize across the last dimension because that dimension stores the embedding features for each token.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

### RoPE (Rotary Position Embeddings)

This is the **big conceptual upgrade** from GPT-2 to Llama, so we will spend extra time here.

#### The problem with learned position embeddings

In GPT-2, we had a position embedding table `wpe` of shape `(seq_len, n_embd)`.
At each position, we looked up a vector and **added** it to the token embedding.
This works, but it has a limitation: the model has never seen positions beyond `seq_len` during training, so it cannot generalize to longer sequences at inference time.

#### The RoPE idea: encode position by rotation

Instead of adding a position vector, RoPE **rotates** the query and key vectors.
The angle of rotation depends on the position in the sequence.

Think of it this way. Take a pair of dimensions `(x0, x1)` from a head vector and treat them as a 2D point.
RoPE applies a 2D rotation matrix:

```
[x0']   [cos(theta)  -sin(theta)] [x0]
[x1'] = [sin(theta)   cos(theta)] [x1]
```

where `theta = position * frequency`.

Each pair of dimensions gets its own frequency. Early pairs rotate fast (high frequency, encoding local position), while later pairs rotate slowly (low frequency, encoding global position). This is analogous to how different digits of a number encode different scales.

#### Why rotation gives you relative position for free

Here is the key mathematical insight. When we compute the attention dot product `Q . K`, the dot product of two rotated vectors depends only on the **difference** in their rotation angles:

```
rotate(q, pos_q) . rotate(k, pos_k) = f(q, k, pos_q - pos_k)
```

This means the attention score between two tokens naturally encodes their **relative distance**, not their absolute positions. The model does not need to learn "position 5 attends to position 3"; it learns "a token attends to the token 2 steps back."

#### The frequency formula

The frequencies are computed as:

`freq_i = 1 / (theta^(2i / head_dim))`

where `i` ranges over dimension pairs and `theta` is the base (default 10000).

- Pair 0: `freq = 1.0` (completes a full rotation every ~6.28 positions)
- Pair d/2 - 1: `freq = 1/theta` (completes a full rotation every ~62,832 positions)

This multi-scale encoding is elegant: nearby positions are distinguished by high-frequency pairs, while distant positions are distinguished by low-frequency pairs.

#### Implementation plan

We split RoPE into two functions:
1. `precompute_rope_freqs`: compute the `(cos, sin)` tables once for all positions.
2. `apply_rope`: use those tables to rotate Q and K inside attention.

The cos/sin tables are shared across all layers (computed once in the model constructor), which is why they are passed into each attention module as registered buffers.

#### `precompute_rope_freqs`

This function builds the cos and sin lookup tables that we will use to rotate Q and K.

Shape: `(seq_len, head_dim // 2)` -- one value per dimension-pair per position.

The computation:
1. Compute the frequency for each dimension pair: `1 / (theta^(2i / head_dim))`.
2. Create a position vector: `[0, 1, 2, ..., seq_len - 1]`.
3. Take the outer product of positions and frequencies to get the rotation angle at each position for each pair.
4. Return `cos(angles)` and `sin(angles)`.

These tables are constant (no learnable parameters) and are computed once, then shared across all layers.

In [ ]:
def precompute_rope_freqs(head_dim: int, seq_len: int, theta: float = 10000.0):
    """Precompute the complex exponentials for RoPE.

    Returns cos and sin tensors of shape (seq_len, head_dim // 2).
    """
    # Frequencies for each dimension pair: theta^(-2i/d) for i in [0, d/2)
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    # Outer product with positions gives the rotation angles
    # positions: [0, 1, 2, ..., seq_len-1]
    # freqs: [theta^0, theta^(-2/d), theta^(-4/d), ...]
    t = torch.arange(seq_len).float()
    angles = torch.outer(t, freqs)  # (seq_len, head_dim // 2)
    return torch.cos(angles), torch.sin(angles)

#### `apply_rope`

This function takes a tensor of shape `(batch, n_head, seq_len, head_dim)` and rotates each pair of dimensions using the precomputed cos/sin tables.

The steps:
1. Split `head_dim` into pairs: reshape the last dimension from `(head_dim,)` to `(head_dim//2, 2)`.
2. Extract the even-index elements `x0` and odd-index elements `x1` from each pair.
3. Apply the 2D rotation: `out0 = x0 * cos - x1 * sin`, `out1 = x0 * sin + x1 * cos`.
4. Interleave back: stack the rotated pairs and flatten to recover the original shape.

Notice that we cast to float32 for the rotation and then cast back.
This prevents precision loss when the model runs in half-precision (float16 or bfloat16).

Also notice: we only apply RoPE to Q and K, **not** to V.
Values do not need position information because they carry the content that gets aggregated; only the attention scores (which come from Q and K) need to be position-aware.

In [ ]:
def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to a tensor.

    Args:
        x: shape (batch, n_head, seq_len, head_dim)
        cos, sin: shape (seq_len, head_dim // 2), precomputed frequencies

    The rotation is applied to pairs of dimensions:
        [x0, x1] -> [x0 * cos - x1 * sin, x0 * sin + x1 * cos]

    This is equivalent to multiplying by a 2D rotation matrix for each pair.
    """
    T = x.shape[2]
    # Split head_dim into pairs: (..., head_dim) -> (..., head_dim//2, 2)
    x_pairs = x.float().reshape(*x.shape[:-1], -1, 2)
    x0 = x_pairs[..., 0]  # even dimensions
    x1 = x_pairs[..., 1]  # odd dimensions

    # Slice cos/sin to match sequence length and reshape for broadcasting
    cos_t = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)
    sin_t = sin[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)

    # Apply 2D rotation to each pair
    out0 = x0 * cos_t - x1 * sin_t
    out1 = x0 * sin_t + x1 * cos_t

    # Interleave back: stack pairs and flatten
    out = torch.stack([out0, out1], dim=-1).flatten(-2)
    return out.type_as(x)

### `GroupedQueryAttention` (GQA)

This is the second big innovation.

#### Recap: standard Multi-Head Attention

In standard MHA (as in our GPT-2 notebook), every attention head has its own Q, K, and V projection.
With `n_head=4` and `head_dim=16`, we have 4 separate Q matrices, 4 separate K matrices, and 4 separate V matrices.

#### The problem: KV-cache during inference

During autoregressive generation, we cache the K and V tensors for all previous tokens so we do not recompute them.
This KV-cache grows linearly with sequence length and is proportional to `n_head * head_dim * 2 * n_layer * seq_len`.
For large models with long contexts, the KV-cache dominates GPU memory.

#### GQA: share K and V across groups of query heads

Grouped Query Attention reduces the KV-cache by using **fewer** K and V heads than Q heads.
Multiple query heads share the same K and V head.

With our config (`n_head=4, n_kv_head=2`):
- Group 0: Q heads 0 and 1 share K0, V0
- Group 1: Q heads 2 and 3 share K1, V1

This cuts the KV-cache in half (and halves the K/V projection parameter count).

Special cases:
- `n_kv_head == n_head`: standard MHA (no sharing, identical to GPT-2).
- `n_kv_head == 1`: Multi-Query Attention (MQA), maximum sharing. All query heads share one K and one V.

#### Implementation detail: `repeat_interleave`

In the forward pass, K and V start with shape `(B, n_kv_head, T, head_dim)` but Q has shape `(B, n_head, T, head_dim)`.
To make the matrix multiplication work, we expand K and V by repeating each KV head `n_groups` times along the head dimension using `repeat_interleave`.
This is a virtual expansion -- PyTorch can optimize the memory layout.

#### RoPE integration

Notice that RoPE is applied to Q and K **after** projection but **before** the dot product.
The cos/sin buffers are registered inside the attention module so they automatically move to the correct device.

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, config: LlamaConfig, cos: torch.Tensor, sin: torch.Tensor):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.n_head % config.n_kv_head == 0

        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_groups = config.n_head // config.n_kv_head  # queries per KV group
        self.head_dim = config.n_embd // config.n_head

        # Q has n_head projections, K and V have n_kv_head (fewer!)
        self.W_q = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.W_k = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.W_v = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.W_o = nn.Linear(config.n_embd, config.n_embd, bias=False)

        self.dropout = nn.Dropout(config.dropout)

        # RoPE frequencies (registered as buffers so they move to the right device)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        # Causal mask
        causal_mask = torch.tril(torch.ones(config.seq_len, config.seq_len))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # Project to Q, K, V
        q = self.W_q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        # q: (B, n_head, T, head_dim)
        # k, v: (B, n_kv_head, T, head_dim)

        # Apply RoPE to Q and K (not V -- values don't need position info)
        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        # Expand K,V to match Q heads by repeating each KV head n_groups times
        # (B, n_kv_head, T, head_dim) -> (B, n_head, T, head_dim)
        if self.n_groups > 1:
            k = k.repeat_interleave(self.n_groups, dim=1)
            v = v.repeat_interleave(self.n_groups, dim=1)

        # Standard attention from here
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_o(out)
        return out

### `SwiGLU` MLP

This is the third innovation.

#### Recap: the standard MLP

In our GPT-2 notebook, the MLP did: `up_project -> activation -> down_project`.
It expanded the hidden dimension by 4x, applied ReLU-squared, and projected back down.

#### The gating idea

SwiGLU splits the "up" step into two parallel projections:
- **Gate**: `Swish(W_gate * x)` -- this learns **what to let through**.
- **Up**: `W_up * x` -- this learns **what to compute**.

The output is their element-wise product:

`SwiGLU(x) = Swish(W_gate * x) * (W_up * x)`

where `Swish(x) = x * sigmoid(x)` (also called SiLU in PyTorch).

Why does gating help? The gate can learn to suppress or amplify specific features, giving the network a richer nonlinearity than a single activation function. Empirically, SwiGLU consistently outperforms ReLU, GELU, and other activations at the same parameter budget.

#### The 2/3 width adjustment

Because we now have **two** up-projections (gate + up) instead of one, we would use 2x the parameters if we kept the same hidden dimension.
To keep the total parameter count roughly the same as a standard 4x MLP, we reduce the hidden dimension:

`hidden_dim = int(4 * n_embd * 2/3)`

This gives `hidden_dim ~= 2.67 * n_embd`. Two projections of this size use about the same parameters as one projection of `4 * n_embd`.

For hardware efficiency, we round this to the nearest multiple of 8.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        # Hidden dim adjusted to keep param count ~same as 4x expansion
        # 4 * n_embd * 2/3 ~ 2.67 * n_embd (but we have 2 up-projections)
        hidden_dim = int(4 * config.n_embd * 2 / 3)
        # Round to nearest multiple of 8 for hardware efficiency
        hidden_dim = ((hidden_dim + 7) // 8) * 8

        self.w_gate = nn.Linear(config.n_embd, hidden_dim, bias=False)  # gate
        self.w_up = nn.Linear(config.n_embd, hidden_dim, bias=False)    # up-projection
        self.w_down = nn.Linear(hidden_dim, config.n_embd, bias=False)  # down-projection
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Gate: learns what to let through
        gate = F.silu(self.w_gate(x))  # Swish = SiLU = x * sigmoid(x)
        # Up: learns what to compute
        up = self.w_up(x)
        # Element-wise product: gated information
        x = gate * up
        # Down-project back to model dimension
        x = self.w_down(x)
        x = self.dropout(x)
        return x

### `LlamaBlock`

A Llama block is structurally identical to a GPT-2 block: two sublayers with residual connections and pre-norm.
1. Attention (now GroupedQueryAttention with RoPE),
2. Feed-forward (now SwiGLU).

The key difference is internal: the attention sublayer uses GQA + RoPE, and the MLP uses SwiGLU.
The residual connection pattern is the same: `x = x + sublayer(norm(x))`.

Notice that `LlamaBlock` takes `cos` and `sin` tensors in its constructor and passes them to the attention layer.
This is how the precomputed RoPE frequencies flow from the top-level model down to each block.

In [ ]:
class LlamaBlock(nn.Module):
    def __init__(self, config: LlamaConfig, cos: torch.Tensor, sin: torch.Tensor):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = GroupedQueryAttention(config, cos, sin)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = SwiGLU(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

### `Llama`

This class assembles the full language model.
Compare this to the GPT-2 model and notice the **one big structural difference**:

**There is no positional embedding table (`wpe`).**

In GPT-2, the forward pass started with `x = wte(idx) + wpe(pos)`.
In Llama, it is simply `x = wte(idx)`.
Position information enters the model through RoPE rotations inside each attention layer, not through an additive embedding at the input.

The forward pass story:
1. Turn token IDs into vectors with the embedding table (`wte`).
2. Pass through a stack of LlamaBlocks (each applying GQA + RoPE + SwiGLU).
3. Normalize one last time with RMSNorm.
4. Project to vocabulary logits with `lm_head`.

Other details:
- RoPE frequencies are precomputed once and passed to every block.
- Residual projection weights (`W_o` in attention, `w_down` in SwiGLU) get smaller initialization, scaled by `1/sqrt(2 * n_layer)`, to keep the residual stream from growing too quickly with depth.
- We use **untied** embeddings: the input embedding table and output projection are separate matrices.

In [ ]:
class Llama(nn.Module):
    def __init__(self, config: LlamaConfig):
        super().__init__()
        self.config = config

        # Token embedding only -- no positional embedding! (RoPE handles position)
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)

        # Precompute RoPE frequencies (shared across all layers)
        head_dim = config.n_embd // config.n_head
        cos, sin = precompute_rope_freqs(head_dim, config.seq_len, config.rope_theta)

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            LlamaBlock(config, cos, sin) for _ in range(config.n_layer)
        ])

        self.ln_f = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Initialize weights
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("W_o.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.seq_len, f"Sequence length {T} exceeds max {self.config.seq_len}"

        # Only token embeddings -- RoPE adds position info inside attention
        x = self.wte(idx)  # (B, T, n_embd)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        print(f"Total parameters: {total:,}")
        print(f"  Token embeddings (wte): {self.wte.weight.numel():,}")
        for i, block in enumerate(self.blocks):
            block_params = sum(p.numel() for p in block.parameters())
            print(f"  Block {i}: {block_params:,}")
        print(f"  Final norm: {sum(p.numel() for p in self.ln_f.parameters()):,}")
        print(f"  LM head:   {self.lm_head.weight.numel():,}")
        return total

## 5. Model Smoke Test

Before training anything, we want a quick confidence check that tensor shapes, gradients, and the GQA parameter savings behave as expected.

What this test demonstrates:
- The model accepts integer token IDs.
- The logits come out with shape `(batch, time, vocab_size)`.
- The random-initialization loss is near `log(256) ~ 5.54`.
- Gradients flow through the entire network.
- **GQA verification**: K projection has fewer parameters than Q projection (the ratio should be `n_kv_head / n_head`).
- **No positional embedding**: the model has no `wpe` attribute (RoPE handles position).

In [ ]:
model = Llama(config)
model.count_parameters()

# Shape check
idx = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(idx)
print(f"\nLogits shape: {logits.shape}")

# Loss check
targets = torch.randint(0, config.vocab_size, (2, 32))
logits, loss = model(idx, targets)
print(f"Random-init loss: {loss.item():.4f}")
print(f"Expected rough baseline: {math.log(config.vocab_size):.4f}")

# Gradient check
loss.backward()
grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"Parameters with gradients: {grad_count}")

# GQA verification: K params should be n_kv_head/n_head of Q params
block = model.blocks[0].attn
q_params = block.W_q.weight.numel()
k_params = block.W_k.weight.numel()
print(f"\nGQA check: Q params={q_params:,}, K params={k_params:,}")
print(f"  K is {config.n_kv_head}/{config.n_head} of Q: {k_params == q_params * config.n_kv_head // config.n_head}")
assert k_params == q_params * config.n_kv_head // config.n_head

# No positional embedding
assert not hasattr(model, "wpe"), "Llama should NOT have learned positional embeddings"
print("No positional embedding table (wpe) -- RoPE handles position!")

## 6. Data Pipeline

The data pipeline is identical to the GPT-2 notebook.
A language model learns from one very long stream of tokens.
Our job is to turn that stream into many smaller `(input, target)` examples.

We use the simplest tokenizer possible: raw UTF-8 bytes.
That keeps the modeling ideas front and center.

### Dataset Paths and Constants

Notebooks are often run from different working directories.
This cell finds the project root in a way that works both when the notebook is opened from the repository root and when it is opened from the `notebook/` folder.

We also define the Tiny Shakespeare download URL here so later functions can stay focused on behavior rather than configuration.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "model.py").exists() and (PROJECT_ROOT.parent / "model.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

### `encode`

`encode` converts text into integer token IDs.
Because we are using bytes, every character becomes one or more integers in the range `0..255`.

Why start with bytes?
- no tokenizer training step,
- no vocabulary-building ceremony,
- easy to inspect and reason about,
- perfect for teaching the language modeling loop.

Trade-off:
byte-level models are simple, but they usually need longer sequences than subword tokenizers to express the same text.

In [ ]:
def encode(text: str) -> list[int]:
    return list(text.encode("utf-8"))

### `decode`

`decode` reverses the tokenizer so we can turn generated token IDs back into readable text.

Teaching note:
`errors="replace"` keeps decoding robust even if the model emits byte combinations that are not valid UTF-8 sequences.
That makes debugging generation easier because the notebook can still display something instead of crashing.

In [ ]:
def decode(tokens: list[int]) -> str:
    return bytes(tokens).decode("utf-8", errors="replace")

### `download_shakespeare`

This function downloads the Tiny Shakespeare dataset once and caches it on disk.
On later runs it simply reads the local file.

Why caching matters in notebooks:
- it keeps reruns fast,
- it avoids repeated network requests,
- it makes the workflow feel deterministic.

In [ ]:
def download_shakespeare() -> str:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    filepath = DATA_DIR / "shakespeare.txt"

    if not filepath.exists():
        print(f"Downloading tiny Shakespeare to {filepath}...")
        response = requests.get(SHAKESPEARE_URL)
        response.raise_for_status()
        filepath.write_text(response.text)
        print(f"Downloaded {len(response.text):,} characters.")

    return filepath.read_text()

### `get_batch`

`get_batch` is the bridge between a giant token stream and minibatch training.
It randomly samples starting positions, slices out windows of length `seq_len`, and creates the shifted targets for next-token prediction.

Shape intuition:
- `x` contains the visible context tokens,
- `y` contains the same sequence shifted one position to the left,
- both end up with shape `(batch_size, seq_len)`.

This one-token shift is the entire supervised learning target for autoregressive language modeling.

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, seq_len: int, device: str = "cpu"):
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in starts])
    y = torch.stack([data[i + 1 : i + 1 + seq_len] for i in starts])
    return x.to(device), y.to(device)

### `prepare_data`

This helper pulls the whole data pipeline together:
1. download or load the raw text,
2. encode it into bytes,
3. turn the bytes into a tensor,
4. split that tensor into train and validation segments.

Teaching choice:
the split is a simple contiguous cut, not a shuffled split.
That keeps the logic easy to read and is good enough for a tiny educational dataset.

In [ ]:
def prepare_data(val_fraction: float = 0.1):
    text = download_shakespeare()
    tokens = encode(text)
    data = torch.tensor(tokens, dtype=torch.long)
    split_idx = int(len(data) * (1 - val_fraction))
    train_data = data[:split_idx]
    val_data = data[split_idx:]
    return train_data, val_data

## 7. Data Smoke Test

We test three things here:
1. tokenizer round-trip correctness,
2. dataset size after preparation,
3. shift-by-one behavior in the training batch.

When teaching, these checks are valuable because many training bugs begin in data handling, not in the model itself.

In [ ]:
sample_text = "Hello, World!"
sample_tokens = encode(sample_text)
round_trip = decode(sample_tokens)
print(sample_tokens)
print(round_trip)
assert round_trip == sample_text

train_data, val_data = prepare_data()
print(f"Total tokens: {len(train_data) + len(val_data):,}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

x, y = get_batch(train_data, batch_size=4, seq_len=32)
print(f"Batch shapes: x={x.shape}, y={y.shape}")
print(f"First input preview:  {decode(x[0].tolist())!r}")
print(f"First target preview: {decode(y[0].tolist())!r}")
assert torch.equal(x[0, 1:], y[0, :-1])

## 8. Training Utilities

The next few functions make the training loop easier to read.
Splitting them out is good software design and good teaching design: each helper introduces one concept at a time.

These are identical to the GPT-2 notebook, adapted only to use `Llama` and `LlamaConfig` where a model class is needed.

### `get_lr`

This function implements a two-stage learning-rate schedule:
1. **warmup**: start small so early updates are stable,
2. **cosine decay**: gradually lower the learning rate so the model can settle into a better solution.

In [ ]:
def get_lr(step: int, max_lr: float, min_lr: float, warmup_steps: int, total_steps: int) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (max_lr - min_lr) * cosine

### `estimate_loss`

Training loss on a single minibatch is noisy.
`estimate_loss` gives a more trustworthy picture by averaging over several fresh batches from both the training split and the validation split.

Important habit:
switch the model to `eval()` mode while measuring and back to `train()` mode afterward.

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, val_data, batch_size, seq_len, device, eval_iters=20):
    model.eval()
    results = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(data, batch_size, seq_len, device)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[name] = sum(losses) / len(losses)
    model.train()
    return results

### `save_checkpoint`

A checkpoint captures the current training state so you can pause work, resume later, or keep the best-performing model.

We save four pieces of information:
- the model weights,
- the optimizer state,
- the config,
- the current step.

In [ ]:
def save_checkpoint(model, config, optimizer, step, path):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "step": step,
        },
        path,
    )

### `load_checkpoint`

This is the inverse of `save_checkpoint`.
It rebuilds the model from the saved config, restores the learned weights, and returns the training step.

Note that we use `Llama` and `LlamaConfig` here, not the GPT-2 classes.

In [ ]:
def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["config"]
    model = Llama(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, config, checkpoint["step"]

### `get_device`

This helper picks the best available accelerator.
It tries CUDA first, then Apple's MPS backend, and falls back to CPU.

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

### `sanity_check`

Before a long training run, it is wise to overfit a single fixed batch.
This is one of the most useful debugging tricks in deep learning.

What success looks like:
the loss should fall very close to zero because the model only needs to memorize one batch.

If this fails, the problem is usually fundamental:
- a bug in the model,
- a bug in the data pipeline,
- a bug in the optimization loop.

In [ ]:
def sanity_check(config, device):
    print("=" * 60)
    print("SANITY CHECK: overfitting one batch")
    print("=" * 60)

    model = Llama(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    train_data, _ = prepare_data()
    x, y = get_batch(train_data, batch_size=4, seq_len=config.seq_len, device=device)

    for step in range(500):
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            print(f"step {step:4d} | loss {loss.item():.4f}")

    final_loss = loss.item()
    print(f"Final loss: {final_loss:.4f}")
    return final_loss < 0.1

### `train`

This is the full training loop.
It combines every previous idea:
- data loading,
- model creation,
- optimizer setup,
- learning-rate scheduling,
- gradient computation,
- gradient clipping,
- validation checks,
- checkpoint saving.

Checkpoints are saved to a `llama` subdirectory inside the checkpoints folder to keep them separate from GPT-2 checkpoints.

As you read it, notice the rhythm of training:
1. choose a learning rate for the current step,
2. sample a batch,
3. run forward and backward passes,
4. update the weights,
5. occasionally evaluate and save.

In [ ]:
def train(
    config,
    device,
    max_steps=2000,
    batch_size=32,
    max_lr=3e-3,
    min_lr=3e-4,
    warmup_steps=100,
    eval_interval=100,
    save_interval=500,
    checkpoint_dir=CHECKPOINT_DIR / "llama",
):
    print("=" * 60)
    print("TRAINING")
    print("=" * 60)
    print(f"Device: {device}")
    print(f"Config: n_embd={config.n_embd}, n_head={config.n_head}, n_kv_head={config.n_kv_head}, n_layer={config.n_layer}")
    print(f"Steps: {max_steps}, Batch size: {batch_size}, Seq len: {config.seq_len}")
    print(f"LR: {max_lr} -> {min_lr} (warmup {warmup_steps} steps)")
    print()

    train_data, val_data = prepare_data()
    model = Llama(config).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    decay_params = [p for p in model.parameters() if p.dim() >= 2]
    nodecay_params = [p for p in model.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": 0.1},
            {"params": nodecay_params, "weight_decay": 0.0},
        ],
        lr=max_lr,
        betas=(0.9, 0.95),
    )

    use_amp = device in ("cuda", "mps")
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    amp_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

    checkpoint_dir = Path(checkpoint_dir)
    os.makedirs(checkpoint_dir, exist_ok=True)
    best_val_loss = float("inf")
    t0 = time.time()

    for step in range(max_steps):
        lr = get_lr(step, max_lr, min_lr, warmup_steps, max_steps)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        x, y = get_batch(train_data, batch_size, config.seq_len, device)

        if use_amp:
            with torch.autocast(device_type=device, dtype=amp_dtype):
                _, loss = model(x, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            _, loss = model(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)

        if step % 10 == 0:
            elapsed = time.time() - t0
            tokens_per_sec = (step + 1) * batch_size * config.seq_len / elapsed if elapsed > 0 else 0
            print(f"step {step:5d} | loss {loss.item():.4f} | lr {lr:.2e} | {tokens_per_sec:.0f} tok/s")

        if step > 0 and step % eval_interval == 0:
            losses = estimate_loss(model, train_data, val_data, batch_size, config.seq_len, device)
            print(f"eval | train {losses['train']:.4f} | val {losses['val']:.4f}")
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                save_checkpoint(model, config, optimizer, step, checkpoint_dir / "best.pt")
                print("saved new best checkpoint")

        if step > 0 and step % save_interval == 0:
            save_checkpoint(model, config, optimizer, step, checkpoint_dir / "latest.pt")

    save_checkpoint(model, config, optimizer, max_steps, checkpoint_dir / "final.pt")
    print(f"Training complete. Best val loss: {best_val_loss:.4f}")
    return model

## 9. Lightweight Pipeline Preview

Instead of launching a full training run right away, this cell does a quick end-to-end pass:
- choose a device,
- build the config and model,
- load data,
- sample one batch,
- compute one forward pass.

This is a great habit in real projects because it catches integration problems early while staying fast enough for iteration.

In [ ]:
device = get_device()
config = LlamaConfig()
model = Llama(config).to(device)
train_data, val_data = prepare_data()
x, y = get_batch(train_data, batch_size=2, seq_len=32, device=device)
logits, loss = model(x, y)

print(f"Selected device: {device}")
print(f"Input batch shape:  {x.shape}")
print(f"Target batch shape: {y.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"One-batch loss:     {loss.item():.4f}")

## 10. Text Generation

Training teaches the model a probability distribution over the next token.
Generation turns that distribution into actual text by repeating the same loop:
1. feed the current context into the model,
2. read the logits for the last position,
3. choose the next token,
4. append it to the context,
5. repeat.

Temperature and top-k are two simple ways to control the sampling behavior.

### `generate`

This function implements autoregressive sampling.

What each argument teaches:
- `prompt_tokens`: generation always starts from some context, even if that context is just a single dummy token.
- `max_new_tokens`: generation is iterative, so we control how many steps to take.
- `temperature`: lower values make the model more conservative; higher values make it more random.
- `top_k`: restricts sampling to the most plausible candidates and reduces wild mistakes.

A subtle but important detail:
we only feed the last `seq_len` tokens back into the model, because the model was trained with a fixed maximum context window.

In [ ]:
@torch.no_grad()
def generate(model, prompt_tokens: list[int], max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 50, device: str = "cpu") -> list[int]:
    model.eval()
    seq_len = model.config.seq_len
    tokens = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx = tokens[:, -seq_len:]
        logits, _ = model(idx)
        logits = logits[:, -1, :]

        if temperature == 0:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_values[:, -1:]] = float("-inf")
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens[0].tolist()

## 11. Optional Checkpoint Demo

This cell tries to load a saved Llama checkpoint and generate text.
If no checkpoint exists yet, it prints a friendly message instead of failing.

That makes the notebook safe to run top-to-bottom in a fresh clone while still showing how inference works once you have trained the model.

In [ ]:
checkpoint_path = CHECKPOINT_DIR / "llama" / "best.pt"

if checkpoint_path.exists():
    demo_model, demo_config, step = load_checkpoint(checkpoint_path, device)
    prompt_tokens = encode("ROMEO:")
    output_tokens = generate(
        demo_model,
        prompt_tokens,
        max_new_tokens=200,
        temperature=0.8,
        top_k=50,
        device=device,
    )
    print(f"Loaded checkpoint from step {step}")
    print(decode(output_tokens))
else:
    print(f"No checkpoint found at {checkpoint_path}.")
    print("Run a training cell first if you want to try generation from learned weights.")

## 12. Exercise

Try these before reading the scaffold cell below:

1. **MHA vs GQA parameter comparison**: Create two configs -- one with `n_kv_head=4` (standard MHA, same as `n_head`) and one with `n_kv_head=1` (MQA, maximum sharing). Build both models and compare total parameter counts. Which attention parameters changed? Which stayed the same?

2. **KV-cache savings**: For a sequence of length 1024, compute the KV-cache size (in number of floats) for MHA (`n_kv_head=4`), GQA (`n_kv_head=2`), and MQA (`n_kv_head=1`). The formula is: `n_kv_head * head_dim * 2 * n_layer * seq_len`.

3. **RoPE theta experiment**: Try changing `rope_theta` from 10000 to 100. How does this affect the rotation frequencies? What happens to the range of positions the model can distinguish?

Reflection question:
Why might GQA hurt quality less than you would expect? Hint: think about what K and V actually represent -- K determines which tokens to attend to, and V determines what information to extract. Do all query heads really need unique copies of that information?

In [ ]:
# Exercise scaffold: MHA vs GQA vs MQA parameter comparison
print("=" * 60)
print("MHA vs GQA vs MQA Parameter Comparison")
print("=" * 60)

mha_config = LlamaConfig(n_kv_head=4)  # Standard MHA: n_kv_head == n_head
gqa_config = LlamaConfig(n_kv_head=2)  # GQA: default
mqa_config = LlamaConfig(n_kv_head=1)  # MQA: maximum sharing

print("\n--- Standard MHA (n_kv_head=4) ---")
mha_model = Llama(mha_config)
mha_total = mha_model.count_parameters()

print("\n--- GQA (n_kv_head=2) ---")
gqa_model = Llama(gqa_config)
gqa_total = gqa_model.count_parameters()

print("\n--- MQA (n_kv_head=1) ---")
mqa_model = Llama(mqa_config)
mqa_total = mqa_model.count_parameters()

print(f"\nParameter savings vs MHA:")
print(f"  GQA: {mha_total - gqa_total:,} fewer params ({(mha_total - gqa_total) / mha_total * 100:.1f}%)")
print(f"  MQA: {mha_total - mqa_total:,} fewer params ({(mha_total - mqa_total) / mha_total * 100:.1f}%)")

# KV-cache comparison
head_dim = 64 // 4  # n_embd // n_head
seq_len_test = 1024
n_layer = 4
print(f"\nKV-cache size for seq_len={seq_len_test}:")
for name, n_kv in [("MHA", 4), ("GQA", 2), ("MQA", 1)]:
    cache_floats = n_kv * head_dim * 2 * n_layer * seq_len_test
    print(f"  {name} (n_kv_head={n_kv}): {cache_floats:,} floats ({cache_floats * 4 / 1024:.1f} KB in float32)")

## 13. Common Pitfalls and Next Steps

### Common Pitfalls
- **Running cells out of order**: later function cells depend on earlier imports and class definitions.
- **Forgetting that Llama has no `wpe`**: if you try to access `model.wpe`, you will get an `AttributeError`. Position is handled by RoPE inside attention.
- **Mismatching `n_head` and `n_kv_head`**: `n_head` must be divisible by `n_kv_head`. If not, the model will raise an assertion error at construction time.
- **Using a sequence length longer than the configured context window**: the model will assert if `T > seq_len`.
- **Confusing token count with character count**: byte tokenization is simple, but some characters map to multiple bytes.
- **Expecting RoPE to handle arbitrary lengths automatically**: while RoPE generalizes better than learned position embeddings, the cos/sin tables are precomputed for `seq_len` positions. To go beyond that, you would need to extend the tables (which is straightforward mathematically but requires code changes).

### Next Steps
- **Sliding Window Attention**: instead of attending to all previous tokens, restrict attention to a fixed window. This bounds memory to O(window_size) instead of O(seq_len) and is used in Mistral.
- **Mixture of Experts (MoE)**: replace the single SwiGLU MLP with a router that sends each token to a subset of expert MLPs. This scales model capacity without proportionally scaling compute. Used in Mixtral and other recent models.
- **KV-cache implementation**: implement the actual KV-cache for efficient autoregressive generation (our `generate` function currently recomputes all keys and values at every step).
- **Tied embeddings**: share weights between `wte` and `lm_head` to reduce parameter count.
- **Longer contexts with RoPE scaling**: experiment with NTK-aware scaling or YaRN to extend the effective context window beyond the training length.